## 6.4 MCS 自适应跟踪

在上一节中，我们验证了安全通信的 FER 性能。本节回到物理层与 MAC 层的交界——在固定 SNR=8 dB 下，让双节点连续发送数据帧，观察 LinkQualityTracker 如何根据 CRC 反馈动态调整 MCS，分析 AMC 的跟踪精度与响应速度。

本节学习大纲如下：

- MCS 自适应跟踪仿真
- MCS 轨迹与 FER 历史曲线
- 跟踪精度分析

### 本实验涉及的关键文件

```
src/nearlink_sdr/
├── node.py               <- SleNode: recommended_mcs / update_mcs
├── mac/
│   └── qos.py             <- LinkQualityTracker: 滑动窗口 FER + MCS 建议
├── common/
│   └── mcs.py             <- get_mcs: MCS 配置查询
├── phy/
│   └── tx_pipeline.py     <- TxConfig: 发射参数 (含 MCS 索引)
└── sim/
    └── link_sim.py        <- sim_dual_node_mcs_adapt: MCS 自适应跟踪
```

---

### 1. MCS 自适应原理

在第四章中我们学习了 LinkQualityTracker 的滑动窗口 FER 统计机制。在双节点场景中，AMC 的闭环流程如下：

- G 节点发送数据帧，T 节点反馈 CRC 结果
- G 节点的 `LinkQualityTracker` 根据滑动窗口 FER（默认 32 帧）计算 MCS 建议
- `recommended_mcs` 自动根据 FER 阈值（< 1% 升，> 10% 降）调整
- G 节点调用 `update_mcs(new_mcs)` 同步更新 TxConfig
- T 节点也需要同步 MCS，确保接收端使用相同的调制和码率解调

在固定 SNR=8 dB 下，MCS=7（QPSK 7/8）的 FER 可能偏高，LinkQualityTracker 会逐步将 MCS 下调到与当前信道条件匹配的等级，体现 AMC 的闭环调节能力。

---

### 2. MCS 跟踪调用链

执行以下函数可以查看完整源码：

In [ ]:
!cat -n src/nearlink_sdr/sim/link_sim.py | sed -n "3861,3941p"

通过调用 SleNode 的三个接口来实现 MCS 的跟踪功能，下面逐一拆解这三个关键接口的调用链。

每帧的完整调用流程：

```
步骤 1: process_feedback(success)   →  将 CRC 结果记入滑动窗口（仅记录，不调整 MCS）
步骤 2: recommended_mcs              →  读取滑动窗口 FER → 判断是否调整 → 输出新 MCS
步骤 3: update_mcs(new_mcs)          →  重建 TxConfig, 同步 G/T 双方
```

**关键理解**：`process_feedback` 负责传入 CRC 结果，MCS 在 `recommended_mcs` 被读取时进行调整——它会根据窗口 FER 决定 ±1/0 并应用到内部状态。

---

**步骤 1：`process_feedback(success)` —— 将 CRC 结果记入滑动窗口**

调用 `LinkQualityTracker.record(crc_ok)` 把本帧 CRC 结果写入滑动窗口 deque。

```python
def process_feedback(self, crc_ok: bool) -> TxDecision:
    return self._qos.on_tx_feedback(crc_ok)
```

`on_tx_feedback` 实际源码（位于 `qos.py`）：

```python
def on_tx_feedback(self, crc_ok: bool) -> TxDecision:
    """处理来自对端的 ACK/NACK 反馈。"""
    self.quality.record(crc_ok)          # 仅记录: CRC 结果写入滑动窗口 deque
    if crc_ok:
        return self.harq.decide_tx(True, self.arq)    # ACK → 发下一帧
    return self.harq.decide_tx(False, self.arq)       # NACK → 准备重传
```

其中 `self.quality` 就是 `LinkQualityTracker` 实例。`record()` 将 bool 追加到滑动窗口，超过 `window_size` 时自动踢掉最旧记录。

---

**步骤 2：`recommended_mcs` —— 读取 FER 并决定是否调整 MCS（核心决策点）**

在此处对 MCS 进行调整。读取属性时，内部调 `suggest_mcs_adjustment()` 根据窗口 FER 返回方向（±1/0），叠加到当前 `config.mcs_index` 并钳位在 0-12。**注意**：`suggest_mcs_adjustment()` 内部已包含 `apply_suggestion()` 调用，即建议值会立即应用到 `LinkQualityTracker._current_mcs`。

```python
@property
def recommended_mcs(self) -> int:
    adj = self._qos.quality.suggest_mcs_adjustment()   # FER < 1% → +1, FER > 10% → -1, 否则 0
    return max(0, min(12, self.config.mcs_index + adj))
```

`suggest_mcs_adjustment()` 内部逻辑（位于 `qos.py`）：
```python
def suggest_mcs_adjustment(self) -> int:
    """建议 MCS 调整方向。

    :returns: +1: 建议提升 MCS (链路质量好)
        -1: 建议降低 MCS (链路质量差)
         0: 保持当前 MCS
    """
    if len(self._history) < self.window_size // 2:
        return 0
     if self.fer < self.fer_target_low and self._current_mcs < 12:
        return 1
    if self.fer > self.fer_target_high and self._current_mcs > 0:
        return -1
    return 0
```

- 窗口不满一半 → 数据不足, 返回 0
- `fer < 0.01` 且 `_current_mcs < 12` → 返回 +1（建议升）
- `fer > 0.10` 且 `_current_mcs > 0` → 返回 -1（建议降）
- 否则返回 0（保持）

---

**步骤 3：`update_mcs(new_mcs)` —— 应用新 MCS，重建发射管线**

将新的 MCS 索引写入配置，并重建 `TxConfig`——调制方式、码率、导频密度等全部按照新 MCS 重新计算。G 和 T 双方必须同步更新，否则 T 会用错误的调制方式解调导致全部失败。

```python
def update_mcs(self, mcs_index: int) -> None:
    self.config.mcs_index = max(0, min(12, mcs_index))
    self._tx_config = TxConfig(
        frame_type=self.config.frame_type,
        mcs_index=self.config.mcs_index,
        pid=self._tx_config.pid,
        ...)  # 调制/码率/导频全部更新
```

---

**仿真中的实际调用**（位于 `sim_dual_node_mcs_adapt`）：

```python
g_node.process_feedback(success)              # 步骤 1: 仅记录 CRC, 不调整 MCS
suggested = g_node.recommended_mcs             # 步骤 2: 读 FER 并根据阈值 (±1/0) 建议新 MCS
if suggested != g_node.config.mcs_index:       # 步骤 3: 值变了才更新
    g_node.update_mcs(suggested)               #         G 节点更新 TxConfig
    t_node.update_mcs(suggested)               #         T 节点必须同步
```

---
### 3. MCS 自适应跟踪仿真

SNR=8 dB，MCS=7（QPSK 7/8）作为起点。连续发送 60 帧，每帧根据 CRC 反馈更新 MCS：

In [ ]:
import sys
sys.path.insert(0, "../src")
import numpy as np
import matplotlib.pyplot as plt
from nearlink_sdr.sim.link_sim import sim_dual_node_mcs_adapt

snr_db = 8.0
n_frames = 60

result = sim_dual_node_mcs_adapt(
    snr_db=snr_db,              # 固定 SNR=8 dB
    n_frames=n_frames,           # 共 60 帧
    initial_mcs=7,               # 从 MCS=7 起步
    seed=42)

mcs_hist = result["mcs_history"]
fer_hist = result["fer_history"]
success_hist = result["success_history"]

print(f"Initial MCS: {mcs_hist[0]}")
print(f"Final MCS:   {mcs_hist[-1]}")
print(f"MCS range:   {min(mcs_hist)} -> {max(mcs_hist)}")
print(f"Times MCS changed: {sum(1 for i in range(1, len(mcs_hist)) if mcs_hist[i] != mcs_hist[i-1])}")
print(f"Final FER (window): {fer_hist[-1]:.3f}")

---

### 4. MCS 轨迹与 FER 历史

左图观察 MCS 的动态调整轨迹，右图观察每帧的 CRC 结果：

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f"MCS Adaptation Tracking (SNR={snr_db:.0f} dB)", fontsize=13)

# 左图: MCS 轨迹
ax1.plot(result["frame_idx"], mcs_hist, "b-", lw=1.5, drawstyle="steps-post")
ax1.set_xlabel("Frame Index"); ax1.set_ylabel("MCS Index")
ax1.set_yticks(range(0, 13))
ax1.set_title(f"MCS Trajectory (init={mcs_hist[0]}, final={mcs_hist[-1]})")
ax1.grid(True, ls="--", alpha=0.5)

# 右图: 每帧成功/失败
ok = [i for i, s in enumerate(success_hist) if s]
fail = [i for i, s in enumerate(success_hist) if not s]
ax2.scatter(ok, [1]*len(ok), c="green", s=12, alpha=0.5, label="CRC OK")
ax2.scatter(fail, [0]*len(fail), c="red", s=12, alpha=0.5, label="CRC FAIL")
ax2.set_xlabel("Frame Index")
ax2.set_yticks([0, 1]); ax2.set_yticklabels(["FAIL", "OK"])
ax2.set_title("Frame CRC Results")
ax2.legend(loc="upper right"); ax2.grid(True, ls="--", alpha=0.3)
plt.tight_layout(); plt.show()

# MCS 对应参数表
from nearlink_sdr.common.mcs import get_mcs
print(f"{'MCS':>4s}  {'Mod':>6s}  {'Rate':>6s}  {'Eff':>6s}")
for m in sorted(set(mcs_hist)):
    mcs = get_mcs(m)
    print(f"{m:4d}  {mcs.modulation.name:>6s}  {str(mcs.code_rate):>6s}  {mcs.spectral_efficiency:6.3f}")

### 5. 实验分析

- **MCS 轨迹**：MCS 从初始值向哪个方向调整？调整了多少次？每次调整的步长是多少
- **调整时机**：MCS 变化是否在连续 CRC 失败/成功之后？与 LinkQualityTracker 的阈值逻辑是否一致
- **收敛性**：MCS 最终是否稳定在某个值？如果未稳定，是仿真帧数不够还是 SNR 恰好处于两个 MCS 的切换边界
- **FER 窗口**：最终滑动窗口 FER 是否落在 1%-10% 的目标范围内

## 课后实践

请补全下方双节点 MCS 自适应调用链中的 **3 处空缺**（每处一行代码），完成 CRC 反馈→MCS 建议→更新 TxConfig 的完整闭环。

要求：

1. 补全 CRC 反馈的传入
2. 补全 MCS 建议的读取
3. 补全新 MCS 的应用（含 T 节点同步）

完成后运行 ，观察 MCS 是否随 CRC 结果动态调整。

In [ ]:
%%writefile mcs_callchain_practice.py
import sys
sys.path.insert(0, "../src")
import numpy as np
from nearlink_sdr.mac.frame import AsyncDataFrame
from nearlink_sdr.mac.link_manager import Role
from nearlink_sdr.node import NodeConfig, NodeRole, SleNode
from nearlink_sdr.sim.link_sim import _channel_impair

snr_db = 8.0
n_frames = 60
rng = np.random.default_rng(42)
g_addr = b"\x01\x02\x03\x04\x05\x06"
t_addr = b"\x0A\x0B\x0C\x0D\x0E\x0F"

g_node = SleNode(config=NodeConfig(
    address=g_addr, role=NodeRole.G_NODE,
    frame_type=2, mcs_index=7))
t_node = SleNode(config=NodeConfig(
    address=t_addr, role=NodeRole.T_NODE,
    frame_type=2, mcs_index=7))
g_node.start_advertising(); g_node.accept_connection(t_addr, Role.G_NODE)
t_node.start_scanning(); t_node.connect(g_addr)

mcs_hist = []
for i in range(n_frames):
    payload = bytes(rng.integers(0, 256, 10, dtype=np.uint8))
    g_node.send(payload)
    tx = g_node.transmit()
    if tx.iq is None:
        g_node._qos.arq.on_ack_received(); continue
    rx_iq = _channel_impair(tx.iq, snr_db, "awgn", 6.0,
        0.0, "none", g_node._tx_config.sps, rng)
    frame = AsyncDataFrame(segment_type=0, data=payload)
    rx = t_node.receive(rx_iq, len(frame.pack()))
    success = rx.success

    # ==== TODO: 补全 MCS 调用链（3处空缺）====
    # TODO 1: 将 CRC 结果传入 LinkQualityTracker
    ______________
    # TODO 2: 读取 MCS 建议
    suggested = ______________
    # TODO 3: 应用新 MCS 并同步 T 节点
    if suggested != g_node.config.mcs_index:
        ______________
        t_node.update_mcs(suggested)

    mcs_hist.append(g_node.config.mcs_index)

print(f"MCS range: {min(mcs_hist)} -> {max(mcs_hist)}")
print(f"Final MCS: {mcs_hist[-1]}")
print(f"Adjustments: {sum(1 for i in range(1, len(mcs_hist)) if mcs_hist[i] != mcs_hist[i-1])}")


执行以下命令进行编译并验证结果：


In [ ]:
!python mcs_callchain_practice.py


执行以下代码获取答案


In [ ]:
!cat answer/06.04_answer.txt
